# Inter-brain neural dynamics — end-to-end demo

Walks through the inter-brain shared-subspace pipeline:

1. Build a `MultiAnimalSession` (or, for offline use, synthetic rates).
2. `fit_shared_subspace` → loadings, time courses, variance partition.
3. Shuffle null + K selection (`choose_n_components`).
4. Time-lagged CCA + cross-animal cell-pair correlation.
5. Per-bin behavior features + `regress_shared_on_behavior`.
6. Six-panel `plot_inter_brain_summary`.

The notebook runs on synthetic data so it executes anywhere. The
real-data block at the bottom is commented out — uncomment when running
from a workstation with the cohort SMB shares mounted.

Reference: Zhang, Phi, Li et al., *Nature* 645, 991–1001 (2025).

In [ ]:
import sys
from pathlib import Path

# Make sibling modules importable from the notebook's location.
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ephys.inter_brain_dynamics import (
    fit_shared_subspace,
    shuffle_null_subspace,
    choose_n_components,
    time_lagged_cca,
    cross_animal_correlation_matrix,
    regress_shared_on_behavior,
)
from ephys.inter_brain_plots import (
    plot_canonical_correlations,
    plot_variance_partition,
    plot_shared_dimensions,
    plot_cross_animal_correlation,
    plot_time_lagged_cca,
    plot_shared_vs_behavior,
    plot_inter_brain_summary,
)

## 1. Synthetic rate matrices

Plant `K_true = 3` shared latent factors plus independent noise in two
animals with different cell counts. Real inputs from
`MultiAnimalSession.get_common_binned_rates` have the same shape
convention: `X = (T, N)` after a `.T` on the per-animal rate matrix.

In [ ]:
rng = np.random.default_rng(0)
T, N_A, N_B, K_true = 1500, 40, 35, 3
Z = rng.standard_normal((T, K_true))
W_A = rng.standard_normal((K_true, N_A))
W_B = rng.standard_normal((K_true, N_B))

X_A = Z @ W_A + 0.5 * rng.standard_normal((T, N_A))
X_B = Z @ W_B + 0.5 * rng.standard_normal((T, N_B))
bin_size_sec = 0.5
bin_centers = np.arange(T) * bin_size_sec

print(f'X_A: {X_A.shape}  (T, N_A)')
print(f'X_B: {X_B.shape}  (T, N_B)')
print(f'Planted shared K = {K_true}')

## 2. Fit the shared subspace

Default `method="regularized"` uses ridge-whitened SVD of the
cross-covariance — robust when N ≳ T.

In [ ]:
fit = fit_shared_subspace(
    X_A, X_B,
    n_components=K_true,
    method='regularized',
    reg=1e-3,
    cv_folds=5,
    animal_ids=('631', '632'),
    bin_size_sec=bin_size_sec,
    t_window=(0.0, T * bin_size_sec),
)
print('train CCs    :', np.round(fit.canonical_correlations['train'], 3))
print('CV-mean CCs  :', np.round(fit.canonical_correlations['cv_mean'], 3))
print('shared_var_A :', f"{fit.variance_partition['shared_var_A_z']:.3f}")
print('shared_var_B :', f"{fit.variance_partition['shared_var_B_z']:.3f}")

## 3. Shuffle null + K selection

`shuffle_null_subspace` uses circular shifts on `X_B` (preserves
per-animal autocorrelation). `choose_n_components` reports two K
recommendations: train-CC > p95-null (the prompt's rule, 5%/dim FPR)
and the sharper CV-mean > p95-null rule.

In [ ]:
selection = choose_n_components(
    X_A, X_B,
    max_K=8,
    n_shuffles=50,
    cv_folds=3,
    seed=0,
)
print('train CCs    :', np.round(selection['train_ccs'], 3))
print('CV-mean CCs  :', np.round(selection['cv_mean'], 3))
print('p95 null     :', np.round(selection['shuffle_p95'], 3))
print(f"recommended_K     (train-CC rule) = {selection['recommended_K']}")
print(f"recommended_K_cv  (CV-mean rule)  = {selection['recommended_K_cv']}")

null = shuffle_null_subspace(
    X_A, X_B, n_components=K_true, n_shuffles=30, seed=0,
)
print(f'\nshuffle null shape: {null.shape}')

## 4. Time-lagged CCA + cross-animal correlation matrix

In [ ]:
lags, ccs = time_lagged_cca(X_A, X_B, max_lag_bins=10, n_components=K_true)
C = cross_animal_correlation_matrix(X_A, X_B)
print(f'lags shape       : {lags.shape}')
print(f'ccs shape        : {ccs.shape}')
print(f'cross-corr shape : {C.shape}')
print(f'peak top-CC lag  : {lags[int(np.nanargmax(ccs[:, 0]))]} bins')

## 5. Behavior regression

For the regression we need per-bin behavior features built per focal
(see `video/behavior_features.py` for the real-data path). Here we
synthesize two feature matrices where animal A's first shared dim is
driven by A's `speed` and dim 1 is driven by B's `speed` — so the
regression should attribute dim 0 to self and dim 1 to partner.

In [ ]:
T_valid = fit.S_A.shape[0]
rng2 = np.random.default_rng(1)
behavior_A = pd.DataFrame({
    'speed': fit.S_A[:, 0] + 0.3 * rng2.standard_normal(T_valid),
    'distance': rng2.standard_normal(T_valid),
})
behavior_B = pd.DataFrame({
    'speed': fit.S_A[:, 1] + 0.3 * rng2.standard_normal(T_valid),
    'distance': rng2.standard_normal(T_valid),
})
regression = regress_shared_on_behavior(
    fit,
    {'631': behavior_A, '632': behavior_B},
    alpha=1.0,
    cv_folds=3,
)
for k in (0, 1):
    r = regression['631'][k]
    print(f"animal 631, k={k+1}: R2_self={r['R2_self']:+.3f}  "
          f"R2_partner={r['R2_partner']:+.3f}  R2_both={r['R2_both']:+.3f}")

## 6. Plots

Each plot function returns a `matplotlib.Figure`; use `plt.show()` /
`fig.savefig(...)` as needed. The six-panel `plot_inter_brain_summary`
is the default report.

In [ ]:
fig_cc = plot_canonical_correlations(fit, shuffle_null=null)
fig_vp = plot_variance_partition(fit)
fig_sd = plot_shared_dimensions(fit, t_bins=bin_centers[fit.valid_mask], k_dims=(0, 1, 2))
fig_xc = plot_cross_animal_correlation(C)
fig_tl = plot_time_lagged_cca(lags, ccs, bin_size_sec=bin_size_sec)
fig_br = plot_shared_vs_behavior(fit, regression)
plt.show()

In [ ]:
fig_summary = plot_inter_brain_summary(
    fit,
    shuffle_null=null,
    t_bins=bin_centers[fit.valid_mask],
    cross_corr=C,
    time_lagged=(lags, ccs),
    regression_results=regression,
    bin_size_sec=bin_size_sec,
)
plt.show()

## Real-data variant (uncomment when SMB shares are mounted)

The same pipeline against a real session — drop-in replacement for the
synthetic `X_A`/`X_B`/`bin_centers`/behavior_* above.

In [ ]:
# from ingestion.multi_animal_session import MultiAnimalSession
# from video.behavior_features import build_behavior_feature_matrix
# from video.tracking_import import load_tracking_data
#
# session = MultiAnimalSession(
#     session_id='20251216',
#     animal_ids=['631', '632'],
#     config_path=None,           # cohort 7 default
# )
# bin_centers, rates_by_animal = session.get_common_binned_rates(
#     bin_size_sec=0.5,
#     smoothing_sigma_sec=0.25,
# )
# X_A = rates_by_animal['631'].T  # (T, N_A)
# X_B = rates_by_animal['632'].T
#
# tracking = load_tracking_data(session.dsm_by_animal[session.sync_from_animal])
# tracking.synchronize_with_ephys(session.sync)
# behavior_A = build_behavior_feature_matrix(
#     tracking, session.events, session.sync, bin_centers,
#     focal='631', partner='632',
# )
# behavior_B = build_behavior_feature_matrix(
#     tracking, session.events, session.sync, bin_centers,
#     focal='632', partner='631',
# )
#
# # Then re-run sections 2–6 verbatim.

Or skip Python and use the CLI:

```bash
python -m ephys.run_inter_brain \
    --session_id 20251216 --animal_ids 631 632 \
    --bin_size 0.5 --smoothing 0.25 \
    --max_K 20 --n_shuffles 200 \
    --behavior_type EC --output_dir ./results
```

Writes `results/inter_brain_20251216_631_632/{results.pkl, summary.png}`.